In [21]:
import numpy as np
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [23]:
with open("data/raw/raw_data.json", "r") as f:
    raw_docs = json.load(f)

In [24]:
documents = [doc["text"] for doc in raw_docs]

In [25]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w.isalnum()]
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

tokenized_docs = [preprocess(doc) for doc in documents]

In [26]:
model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

model.save("models/embeddings/word2vec.model")

In [27]:
def get_vector(tokens):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(100)
    
    return np.mean(vectors, axis=0)

doc_embeddings = np.array([
    get_vector(doc) for doc in tokenized_docs
])

np.save("models/embeddings/doc_embeddings_w2v.npy", doc_embeddings)

In [28]:
def query_to_vec(query):
    tokens = preprocess(query)
    return get_vector(tokens)

In [29]:
def search(query, top_k=5):
    
    q_vec = query_to_vec(query)
    
    scores = cosine_similarity([q_vec], doc_embeddings)[0]
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    
    for idx in top_indices:
        results.append({
            "title": raw_docs[idx]["title"],
            "text": raw_docs[idx]["text"][:150],
            "score": float(scores[idx])
        })
    
    return results

In [31]:
with open("queries.txt", "r") as f:
    queries = [line.strip() for line in f.readlines()]

for q in queries:
    print("\n" + "="*50)
    print("Query:", q)
    
    results = search(q)
    
    for i, r in enumerate(results, 1):
        print(f"\nRank {i}")
        print("Title:", r["title"])
        print("Score:", round(r["score"], 4))
        print("Text:", r["text"], "...")


Query: food delivery problem

Rank 1
Title: Junk food
Score: 0.8665
Text: {{no sources}}
[[File:USMC-100629-M-3215R-002.jpg|thumb|200px|A poster at Camp Pendleton's 21-Area Health Promotion Center shows the effects of junk f ...

Rank 2
Title: Food security
Score: 0.8535
Text: {{more sources}}
[[File:Africa_Food_Security_18_(10665134354).jpg|thumb|A woman selling [[produce]] at a market in [[Lilongwe]], [[Malawi]]]]
[[File:A ...

Rank 3
Title: Food
Score: 0.8531
Text: {{more sources|date=August 2014}}
[[File:Good Food Display - NCI Visuals Online.jpg|thumb|200px|Different kinds of foods.]]
[[File:Joachim_Beuckelaer_ ...

Rank 4
Title: Ultra-processed food
Score: 0.8477
Text: '''Ultra processed food''' is a classification in the NOVA food classification system. 

The NOVA system was developed by the [[Brazilian]] [[nutritio ...

Rank 5
Title: Fast food
Score: 0.846
Text: {{multiple image
| perrow = 2
| total_width = 300
| image1 = Cheeseburger.png
| image2 = Tumbler of cola with ice.jp

In [32]:
def evaluate(results, k=5):
    relevant = 0
    
    for i, r in enumerate(results[:k], 1):
        print(f"\nRank {i}: {r['title']}")
        label = int(input("Relevant? (1/0): "))
        relevant += label
    
    return relevant / k

In [33]:
scores = []

for q in queries:
    print("\n" + "="*50)
    print("Evaluating:", q)
    
    results = search(q)
    
    p = evaluate(results)
    scores.append(p)
    
    print("Precision@5:", round(p, 2))

print("\nAverage Precision@5:", sum(scores)/len(scores))


Evaluating: food delivery problem

Rank 1: Junk food

Rank 2: Food security

Rank 3: Food

Rank 4: Ultra-processed food

Rank 5: Fast food
Precision@5: 1.0

Evaluating: climate change effects

Rank 1: Glossary of climate change

Rank 2: Semi-arid climate

Rank 3: Climate commitment studies

Rank 4: Köppen climate classification

Rank 5: Oceanic climate
Precision@5: 1.0

Evaluating: machine learning applications

Rank 1: Computer assisted language learning

Rank 2: Computer engineering

Rank 3: Event-driven programming

Rank 4: Computer hardware

Rank 5: Implementation
Precision@5: 0.6

Evaluating: health benefits of exercise

Rank 1: Estonian Health Insurance Fund

Rank 2: Cardiac rehabilitation

Rank 3: Nutritionist

Rank 4: Healthy lifestyle

Rank 5: Public health
Precision@5: 1.0

Evaluating: economic inflation causes

Rank 1: Economic bubble

Rank 2: Fiscal policy

Rank 3: Consumer confidence

Rank 4: Social issue

Rank 5: Macroeconomics
Precision@5: 1.0

Average Precision@5: 0.91